In [5]:
import pandas as pd

In [6]:
df = pd.read_csv("Megastore Dataset.csv")

In [7]:
df.head(10)

,OrderID,ProductName,Quantity,InvoiceDate,UnitPrice,TotalCost,Country,DiscountApplied,OrderPriority,Region,Segment,ExpeditedShipping,PaymentMethod,CustomerOrderSatisfaction
0,536370,INFLATABLE POLITICAL GLOBE,48,12/1/2010 8:45,$0.85,$40.80,United States,Yes,High,Northeast,Corporate,Yes,Credit Card,Satisfied
1,536370,SET2 RED RETROSPOT TEA TOWELS,18,12/1/2010 8:45,$2.95,$53.10,United States,Yes,High,Northeast,Corporate,Yes,Credit Card,Satisfied
2,536370,PANDA AND BUNNIES STICKER SHEET,12,12/1/2010 8:45,$0.85,$10.20,United States,Yes,High,Northeast,Corporate,Yes,Credit Card,Satisfied
3,536370,RED TOADSTOOL LED NIGHT LIGHT,24,12/1/2010 8:45,$1.65,$39.60,United States,Yes,High,Northeast,Corporate,Yes,Credit Card,Satisfied
4,536370,VINTAGE HEADS AND TAILS CARD GAME,24,12/1/2010 8:45,$1.25,$30.00,United States,Yes,High,Northeast,Corporate,Yes,Credit Card,Satisfied
5,536370,STARS GIFT TAPE,24,12/1/2010 8:45,$0.65,$15.60,United States,Yes,High,Northeast,Corporate,Yes,Credit Card,Satisfied
6,536370,VINTAGE SEASIDE JIGSAW PUZZLES,12,12/1/2010 8:45,$3.75,$45.00,United States,Yes,High,Northeast,Corporate,Yes,Credit Card,Satisfied
7,536370,ROUND SNACK BOXES SET OF4 WOODLAND,24,12/1/2010 8:45,$2.95,$70.80,United States,Yes,High,Northeast,Corporate,Yes,Credit Card,Satisfied
8,536370,MINI PAINT SET VINTAGE,36,12/1/2010 8:45,$0.65,$23.40,United States,Yes,High,Northeast,Corporate,Yes,Credit Card,Satisfied
9,536370,MINI JIGSAW CIRCUS PARADE,24,12/1/2010 8:45,$0.42,$10.08,United States,Yes,High,Northeast,Corporate,Yes,Credit Card,Satisfied


In [8]:
item_count = df.groupby('OrderID')['ProductName'].count().reset_index(name='item_count')
print(item_count)

     OrderID  item_count
0     536370          19
1     536852           6
2     536974          15
3     537065          62
4     537463          40
..       ...         ...
436   581001          18
437   581171          19
438   581279           2
439   581316           3
440   581587          15

[441 rows x 2 columns]


In [9]:
orderid = 536370
items = df[df['OrderID']==orderid]['ProductName'].unique().tolist()
print(orderid, items)

536370 ['INFLATABLE POLITICAL GLOBE ', 'SET2 RED RETROSPOT TEA TOWELS ', 'PANDA AND BUNNIES STICKER SHEET', 'RED TOADSTOOL LED NIGHT LIGHT', 'VINTAGE HEADS AND TAILS CARD GAME ', 'STARS GIFT TAPE ', 'VINTAGE SEASIDE JIGSAW PUZZLES', 'ROUND SNACK BOXES SET OF4 WOODLAND ', 'MINI PAINT SET VINTAGE ', 'MINI JIGSAW CIRCUS PARADE ', 'MINI JIGSAW SPACEBOY', 'SPACEBOY LUNCH BOX ', 'CIRCUS PARADE LUNCH BOX ', 'LUNCH BOX I LOVE LONDON', 'CHARLOTTE BAG DOLLY GIRL DESIGN', 'ALARM CLOCK BAKELIKE GREEN', 'ALARM CLOCK BAKELIKE RED ', 'ALARM CLOCK BAKELIKE PINK', 'SET 2 TEA TOWELS I LOVE LONDON ']


Ordinal Encoding

In [10]:
#Confirm categories
print("OrderPriority categories:", sorted(df["OrderPriority"].dropna().unique()))
print("CustomerOrderSatisfaction categories:", sorted(df["CustomerOrderSatisfaction"].dropna().unique()))
print("Region categories:", sorted(df["Region"].dropna().unique()))
print("PaymentMethod categories:", sorted(df["PaymentMethod"].dropna().unique()))

OrderPriority categories: ['High', 'Medium']
CustomerOrderSatisfaction categories: ['Dissatisfied', 'Prefer not to answer', 'Satisfied', 'Very Dissatisfied', 'Very Satisfied']
Region categories: ['Northeast', 'Southeast']
PaymentMethod categories: ['Credit Card', 'PayPal']


In [12]:
#Define Ordinal for OrderPriority and CustomerOrderSatisfaction

order_priority_map = {
    "Medium": 1,
    "High": 2
}

df["OrderPriority_High_encoded"] = df["OrderPriority"].map(order_priority_map)

# CustomerOrderSatisfaction: explicit ordinal ranking starting at 1
customer_satisfaction_map = {
    "Very Dissatisfied": 1,
    "Dissatisfied": 2,
    "Prefer not to answer": 3,
    "Satisfied": 4,
    "Very Satisfied": 5
}

df["CustomerOrderSatisfaction_VerySatisfied_encoded"] = (
    df["CustomerOrderSatisfaction"].map(customer_satisfaction_map)
)

One-Hot Encoding

In [13]:
#Apply one-hot encoding to NOMINAL variables
df_encoded = pd.get_dummies(df, columns=["Region", "PaymentMethod"])

In [15]:
# Check that your new columns exist
print("\nEncoded columns created:")
print([c for c in df_encoded.columns if "encoded" in c.lower()][:10])


Encoded columns created:
['OrderPriority_High_encoded', 'CustomerOrderSatisfaction_VerySatisfied_encoded']


In [16]:
#Export dataset with encoded variables
df_encoded.to_csv("encoded_dataset.csv", index=False)
print("Export complete: encoded_dataset.csv")

print(df_encoded[[
    "OrderPriority",
    "OrderPriority_High_encoded",
    "CustomerOrderSatisfaction",
    "CustomerOrderSatisfaction_VerySatisfied_encoded"
]].head(10))


Export complete: encoded_dataset.csv
  OrderPriority  OrderPriority_High_encoded CustomerOrderSatisfaction  \
0          High                           2                 Satisfied   
1          High                           2                 Satisfied   
2          High                           2                 Satisfied   
3          High                           2                 Satisfied   
4          High                           2                 Satisfied   
5          High                           2                 Satisfied   
6          High                           2                 Satisfied   
7          High                           2                 Satisfied   
8          High                           2                 Satisfied   
9          High                           2                 Satisfied   

   CustomerOrderSatisfaction_VerySatisfied_encoded  
0                                                4  
1                                                4  


In [17]:
#Load the dataset and build a transactional binary matrix

basket = (
    df.groupby(["OrderID", "ProductName"])["Quantity"]
      .sum()
      .unstack(fill_value=0)
)

In [18]:
# Convert to binary (Apriori/FP-Growth need presence/absence)

basket = (basket > 0).astype(int)

In [64]:
print("Basket shape (orders, products):", basket.shape)
print(basket.iloc[:5, :10])

Basket shape (orders, products): (441, 1562)
ProductName   50S CHRISTMAS GIFT BAG LARGE   DOLLY GIRL BEAKER  \
OrderID                                                          
536370                                   0                   0   
536852                                   0                   0   
536974                                   0                   0   
537065                                   0                   0   
537463                                   0                   0   

ProductName   I LOVE LONDON MINI BACKPACK   NINE DRAWER OFFICE TIDY  \
OrderID                                                               
536370                                  0                         0   
536852                                  0                         0   
536974                                  0                         0   
537065                                  0                         0   
537463                                  0                         

In [65]:
#Show one orders actual items

order_id = 536370 
items_in_order = basket.columns[basket.loc[order_id] == 1].tolist()
print(f"Order {order_id} contains {len(items_in_order)} items:")
for i in items_in_order[:30]:
    print("-", i)

Order 536370 contains 19 items:
- ALARM CLOCK BAKELIKE GREEN
- ALARM CLOCK BAKELIKE PINK
- ALARM CLOCK BAKELIKE RED 
- CHARLOTTE BAG DOLLY GIRL DESIGN
- CIRCUS PARADE LUNCH BOX 
- INFLATABLE POLITICAL GLOBE 
- LUNCH BOX I LOVE LONDON
- MINI JIGSAW CIRCUS PARADE 
- MINI JIGSAW SPACEBOY
- MINI PAINT SET VINTAGE 
- PANDA AND BUNNIES STICKER SHEET
- RED TOADSTOOL LED NIGHT LIGHT
- ROUND SNACK BOXES SET OF4 WOODLAND 
- SET 2 TEA TOWELS I LOVE LONDON 
- SET2 RED RETROSPOT TEA TOWELS 
- SPACEBOY LUNCH BOX 
- STARS GIFT TAPE 
- VINTAGE HEADS AND TAILS CARD GAME 
- VINTAGE SEASIDE JIGSAW PUZZLES


In [66]:
# Export transactional dataset

basket.to_csv("transaction_dataset.csv")
print("Export complete: transaction_dataset.csv")

Export complete: transaction_dataset.csv


In [67]:
#Generate association rules wiht the Apriori algorithm

from mlxtend.frequent_patterns import apriori, association_rules

# Frequent itemsets
freq = apriori(basket, min_support=0.01, use_colnames=True)
print("Frequent itemsets found:", len(freq))

# Rules
rules = association_rules(freq, metric="lift", min_threshold=1.0)
print("Rules found:", len(rules))

# Sort and show Top 3
rules_sorted = rules.sort_values(["lift", "confidence", "support"], ascending=False)

top3 = rules_sorted.head(3)[["antecedents","consequents","support","confidence","lift"]]
print("\nTOP 3 RULES:")
print(top3)

/Users/janekaporter/anaconda3/lib/python3.11/site-packages/mlxtend/frequent_patterns/fpcommon.py:175: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


Frequent itemsets found: 8340
Rules found: 85174

TOP 3 RULES:
                                             antecedents  \
4760                      (SMALL MARSHMALLOWS PINK BOWL)   
4761                (SMALL DOLLY MIX DESIGN ORANGE BOWL)   
53650  (ALARM CLOCK BAKELIKE PINK, PLASTERS IN TIN CI...   

                                             consequents   support  \
4760                (SMALL DOLLY MIX DESIGN ORANGE BOWL)  0.011338   
4761                      (SMALL MARSHMALLOWS PINK BOWL)  0.011338   
53650  (PLASTERS IN TIN SPACEBOY, ALARM CLOCK BAKELIK...  0.011338   

       confidence  lift  
4760          1.0  88.2  
4761          1.0  88.2  
53650         1.0  88.2  
